<a href="https://colab.research.google.com/github/rfandan/Transformers/blob/main/Modern_MHA_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Simple sentence
# input:  "the cat walked over the road"
# target: "cat walked over the road <eos>"
text = "The cat walked over the road"

# Build vocab
tokens = text.lower().split()
vocab = {w: i for i, w in enumerate(sorted(set(tokens)))}
inv_vocab = {i: w for w, i in vocab.items()}

# Encode
x_tokens = torch.tensor([vocab[w] for w in tokens])  # (T,)
x = x_tokens.unsqueeze(0)  # (B=1, T)

# shift
inputs = x[:, :-1]
targets = x[:, 1:]

vocab_size = len(vocab)

# Embedding layer
d_model = 16
n_heads = 2
head_dim = d_model // n_heads

embedding = nn.Embedding(vocab_size, d_model)

x = embedding(x_tokens)  # (B, T, d_model)

# RoPE (Rotary Positional Embeddings)
def rotate_every_two(x):
    # split into pairs: (x1, x2)
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]

    # rotate: (-x2, x1)
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def rope_angles(seq_len, dim, device):
    half = dim // 2

    # frequencies
    freq = 1.0 / (10000 ** (torch.arange(half, device=device) / half))

    # positions
    pos = torch.arange(seq_len, device=device)

    # outer product → (T, D/2)
    angles = torch.einsum("t,d->td", pos, freq)

    cos = angles.cos()[None, :, None, :]  # (1, T, 1, D/2)
    sin = angles.sin()[None, :, None, :]

    return cos, sin


def apply_rope(x, cos, sin):
    # x: (B, T, H, D)
    # cos, sin: (1, T, 1, D/2)

    # Split the last dimension into two halves for real and imaginary parts
    x_real = x[..., 0::2] # (B, T, H, D/2)
    x_imag = x[..., 1::2] # (B, T, H, D/2)

    # Apply the rotation formula
    out_real = x_real * cos - x_imag * sin
    out_imag = x_real * sin + x_imag * cos

    # Interleave the real and imaginary parts
    return torch.stack((out_real, out_imag), dim=-1).flatten(-2)

In [28]:
# Multi-Head Attention
class MHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        # ---- 1. Project to Q, K, V ----
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        # ---- 2. Reshape into heads ----
        q = q.view(B, T, self.num_heads, self.head_dim)
        k = k.view(B, T, self.num_heads, self.head_dim)
        v = v.view(B, T, self.num_heads, self.head_dim)

        # ---- 3. Apply RoPE (only Q, K) ----
        cos, sin = rope_angles(T, self.head_dim, x.device)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # ---- 4. Rearrange for attention ----
        q = q.transpose(1, 2)  # (B, H, T, D)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ---- 5. Scaled dot-product attention ----
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # ---- 6. Merge heads ----
        attn = attn.transpose(1, 2).contiguous().view(B, T, C)

        # ---- 7. Output projection ----
        return self.out(attn)

In [29]:
# Top-2 MoE (with capacity + aux loss)
class Top2MoE(nn.Module):
    def __init__(self, d_model, hidden_dim, num_experts, capacity_factor=1.25):
      super().__init__()
      self.num_experts = num_experts
      self.capacity_factor = capacity_factor

      # Router: decides which expert to use
      self.router = nn.Linear(d_model, num_experts, bias=False)

      # Experts: independent MLPs
      self.experts = nn.ModuleList([
          nn.Sequential(
              nn.Linear(d_model, hidden_dim),
              nn.SiLU(),
              nn.Linear(hidden_dim, d_model)
          )
          for _ in range(num_experts)
      ])

    def forward(self,x):
      B, T, C =x.shape
      N=B*T
      # ---- 1. Flatten tokens ----
      x_flat = x.view(N, C)
      # ---- 2. Routing ----
      logits = self.router(x_flat) # (N, E)
      probs = F.softmax(logits, dim=-1) # (N, E)
      # ---- 3. Top-2 experts per token ----
      top2_vals, top2_idx = torch.topk(probs, k=2, dim=-1)
      # normalize weights
      top2_vals = top2_vals / top2_vals.sum(dim=-1, keepdim=True)
      # ---- 4. Capacity ----
      capacity = int(self.capacity_factor * N / self.num_experts)

      out = torch.zeros_like(x_flat)
      expert_counts = torch.zeros(self.num_experts, device=x.device)

      # ---- 5. Dispatch tokens to experts ----
      for expert_id in range(self.num_experts):
        mask = (top2_idx == expert_id)
        if not mask.any():
          continue

        token_idx, which_slot = mask.nonzero(as_tuple=True)
        # enforce capacity
        if token_idx.numel() > capacity:
          token_idx = token_idx[:capacity]
          which_slot = which_slot[:capacity]

        expert_counts[expert_id] = token_idx.numel()
        selected_x = x_flat[token_idx]
        # process tokens in batch
        expert_out = self.experts[expert_id](selected_x)
        weights = top2_vals[token_idx, which_slot].unsqueeze(-1)
        out[token_idx] += weights * expert_out

      # ---- 6. Load balancing loss ----
      importance = probs.sum(dim=0)
      load = expert_counts
      importance = importance / importance.sum()
      load = load / load.sum()

      aux_loss = (importance * load).sum() * self.num_experts

      return out.view(B, T, C), aux_loss, top2_idx.view(B, T, 2)

In [30]:
# Transformer Block (pre-norm)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, num_experts):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attn = MHA(d_model, num_heads)

        self.norm2 = nn.RMSNorm(d_model)
        self.moe = Top2MoE(d_model, hidden_dim, num_experts)

    def forward(self, x):
        # Attention + residual
        x = x + self.attn(self.norm1(x))

        # MoE + residual
        moe_out, aux_loss, routing = self.moe(self.norm2(x))
        x = x + moe_out

        return x, aux_loss, routing

In [31]:
# Full Mini Model
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, hidden_dim, num_experts):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)

        self.block = TransformerBlock(
            d_model, num_heads, hidden_dim, num_experts
        )

        self.norm = nn.RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tokens):
        x = self.embed(tokens)

        x, aux_loss, routing = self.block(x)

        x = self.norm(x)

        logits = self.head(x)

        return logits, aux_loss, routing

In [32]:
model = MiniTransformer(
    vocab_size=len(vocab),
    d_model=16,
    num_heads=2,
    hidden_dim=32,
    num_experts=4
)

In [33]:
# Simple training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(200):

    logits, aux_loss, routing = model(inputs)

    loss_main = F.cross_entropy(
        logits.view(-1, len(vocab)),
        targets.view(-1)
    )

    loss = loss_main + 0.01 * aux_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

step 0 | loss 1.5965
step 20 | loss 0.0626
step 40 | loss 0.0172
step 60 | loss 0.0135
step 80 | loss 0.0125
step 100 | loss 0.0120
step 120 | loss 0.0116
step 140 | loss 0.0114
step 160 | loss 0.0112
step 180 | loss 0.0110


In [34]:
# Visualizing expert usage
def print_routing(tokens, routing):
    tokens = tokens[0]
    routing = routing[0]  # (T, 2)

    for i, token_id in enumerate(tokens):
        word = inv_vocab[token_id.item()]
        experts = routing[i].tolist()

        print(f"{word:>10} → experts {experts}")

In [35]:
logits, aux_loss, routing = model(inputs)

print_routing(inputs, routing)

       the → experts [0, 2]
       cat → experts [0, 1]
    walked → experts [3, 1]
      over → experts [2, 3]
       the → experts [0, 2]


In [36]:
def expert_usage(routing, num_experts):
    counts = torch.zeros(num_experts)

    flat = routing.view(-1)

    for i in flat:
        counts[i] += 1

    return counts

In [37]:
counts = expert_usage(routing, num_experts=4)
print("Expert usage:", counts)

Expert usage: tensor([3., 2., 3., 2.])
